In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import pytz

# ---------- PAGE TITLE ----------
print("📊 Google Play Store – App Category Dashboard")

# ---------- TIME CHECK (IST) ----------
ist = pytz.timezone("Asia/Kolkata")
now = datetime.now(ist)

start_time = now.replace(hour=15, minute=0, second=0)
end_time = now.replace(hour=17, minute=0, second=0)

# ---------- LOAD DATA ----------
df = pd.read_csv("Play Store Data.csv")

# ---------- CLEAN COLUMN NAMES ----------
df.columns = df.columns.str.strip()

# ---------- CLEAN DATA ----------
df['Installs'] = df['Installs'].str.replace('[+,]', '', regex=True)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

df['Size'] = df['Size'].replace('Varies with device', None)
df['Size'] = df['Size'].str.replace('M', '', regex=False)
df['Size'] = pd.to_numeric(df['Size'], errors='coerce')

df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

df = df.dropna(subset=['Installs', 'Reviews', 'Rating', 'Size', 'Last Updated'])

df['Installs'] = df['Installs'].astype(int)
df['Reviews'] = df['Reviews'].astype(int)

# ---------- TIME CONDITION ----------
if start_time <= now <= end_time:
    print("🕒 Graph visible only between 3 PM – 5 PM IST")

    # ---------- FILTER ----------
    filtered = df[
        (df['Rating'] >= 4.0) &
        (df['Size'] >= 10) &
        (df['Last Updated'].dt.month == 1)
    ]

    # ---------- AGGREGATION ----------
    top10 = (
        filtered.groupby('Category')
        .agg({
            'Installs': 'sum',
            'Rating': 'mean',
            'Reviews': 'sum'
        })
        .sort_values(by='Installs', ascending=False)
        .head(10)
    )

    categories = top10.index
    avg_ratings = top10['Rating']
    total_reviews = top10['Reviews']

    x = np.arange(len(categories))
    width = 0.35

    # ---------- PLOT ----------
    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax2 = ax1.twinx()

    # 🟠 Total Reviews (main volume)
    ax1.bar(
        x - width/2,
        total_reviews,
        width,
        label="Total Reviews",
        color="#ff7f0e",
        alpha=0.9,
        edgecolor="black",
        zorder=2
    )

    # 🔵 Average Rating (secondary metric)
    ax2.bar(
        x + width/2,
        avg_ratings,
        width,
        label="Average Rating",
        color="#1f77b4",
        alpha=0.9,
        edgecolor="black",
        zorder=3
    )

    ax1.set_xlabel("App Category")
    ax1.set_ylabel("Total Reviews")
    ax2.set_ylabel("Average Rating")

    ax1.set_title("Top 10 App Categories by Installs")

    ax1.set_xticks(x)
    ax1.set_xticklabels(categories, rotation=45, ha="right")

    # Set rating scale clearly
    ax2.set_ylim(0, 5)

    # Legends
    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")

    # ---------- FILTER NOTE ----------
    plt.figtext(
        0.5, -0.15,
        "Filters applied: Rating ≥ 4.0 | Size ≥ 10M | Last Updated in January",
        ha="center",
        fontsize=9
    )

    plt.tight_layout()
    plt.show()

else:
    print("⏰ Dashboard is accessible only between 3 PM – 5 PM IST")


📊 Google Play Store – App Category Dashboard
⏰ Dashboard is accessible only between 3 PM – 5 PM IST
